# Trascrivi Lezioni — Google Colab

Turn your lectures (audio/video **files** or **links** like YouTube/Webex) into
cleaned, translated **Word documents** — entirely in the cloud, nothing to install.
Powered by Google Gemini.

**How to use it — just run the cells top to bottom (▶):**

1. **Step 1 — Install**: run it once (~1 min).
2. **Step 2 — API key**: get a free key at https://aistudio.google.com/apikey .
   Best: store it as a Colab **Secret** (left sidebar → 🔑 key icon → *Add new secret*,
   name it `GOOGLE_API_KEY`, turn on *Notebook access*). Otherwise the cell asks you to paste it.
3. **Step 3 — Add lectures**: click **Choose Files** and pick your videos (and, for
   Webex, a `cookies.txt`). You can also just drag files into the **📁 Files** panel on
   the left — the notebook finds them automatically. No need to type any file name.
4. **Step 4 — Options**: fill the form (paste any **links**, choose languages).
5. **Step 5 — Transcribe**: run it. At the end your Word files download as a `.zip`.

---
**Privacy:** your audio/transcripts are sent to Google Gemini for processing. Only
transcribe recordings you are allowed to use. A free CPU runtime is enough.


In [ ]:
#@title Step 1 — Install (run once) { display-mode: "form" }
# Installs the transcriber + yt-dlp in this Colab session. ffmpeg is already on Colab.
!pip install -q "git+https://github.com/GiorgioAlo/lecture-transcriber"
print("Installed. Move on to Step 2.")


In [ ]:
#@title Step 2 — Load your Gemini API key { display-mode: "form" }
import os
try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    print("API key loaded from Colab Secrets.")
except Exception:
    import getpass
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Paste your Gemini API key (hidden): ")
    print("API key set for this session.")


In [ ]:
#@title Step 3 — Add your lectures (upload files) { display-mode: "form" }
# Click "Choose Files" and pick your lecture videos/audio (mp4, m4a, ...).
# For Webex links, also upload a cookies.txt here.
# (Alternative: drag files into the Files panel on the left — they are found too.)
from google.colab import files
uploaded = files.upload()
if uploaded:
    print("Uploaded:", list(uploaded.keys()))
else:
    print("Nothing uploaded here. If you dragged files into /content, that is fine too.")


In [ ]:
#@title Step 4 — Options { display-mode: "form" }
#@markdown Paste one or more **links** (YouTube/Webex), separated by spaces. Leave empty if you only use uploaded files.
LINKS = "" #@param {type:"string"}
#@markdown Output document **languages** (space-separated, e.g. `it en` or `it en fr`):
LANGUAGES = "it en" #@param {type:"string"}
#@markdown Optional **course** name (helps cleanup/translation):
COURSE = "" #@param {type:"string"}
print("Links:", LINKS or "(none)", "| Languages:", LANGUAGES, "| Course:", COURSE or "(none)")


In [ ]:
#@title Step 5 — Transcribe (then download the .zip) { display-mode: "form" }
import os, shutil
from pathlib import Path
from trascrittore import Job, esegui, config
from trascrittore.progress import ReporterConsole

key = os.environ.get("GOOGLE_API_KEY", "")
assert key, "Run Step 2 (API key) first."

OUTPUT_DIR = "/content/transcriptions"
WORK = "/content/.cache"
MEDIA = (".mp4", ".m4a", ".mp3", ".wav", ".aac", ".ogg", ".flac", ".mkv", ".mov", ".webm")

content = Path("/content")
file_lezioni = sorted(p for p in content.iterdir() if p.is_file() and p.suffix.lower() in MEDIA)
links = [u for u in LINKS.replace(",", " ").split() if u.lower().startswith(("http://", "https://"))]
cookies = "/content/cookies.txt" if Path("/content/cookies.txt").exists() else None
langs = LANGUAGES.split() or ["it", "en"]

sources = [("file", str(p)) for p in file_lezioni] + [("link", u) for u in links]
if not sources:
    raise SystemExit("No lectures found. Upload files in Step 3 (or drag them into /content), or add links in Step 4.")

print("Lectures to process:")
for kind, s in sources:
    print("  -", Path(s).name if kind == "file" else s)

out = Path(OUTPUT_DIR); out.mkdir(parents=True, exist_ok=True)
saved, failed = [], 0
for i, (kind, src) in enumerate(sources, 1):
    name = Path(src).stem if kind == "file" else "Lecture"
    print()
    print(f"=== [{i}/{len(sources)}] {name} ===")
    job = Job(
        sorgente=Path(src) if kind == "file" else None,
        url=src if kind == "link" else None,
        nome=name, lingue_output=langs, corso=COURSE, cookies=cookies,
    )
    try:
        ris = esegui(job, key, ReporterConsole(config.FASI), Path(WORK))
    except Exception as e:
        print("  [x] failed:", e); failed += 1; continue
    for d in ris.documenti:
        dest = out / Path(d).name
        shutil.copy2(d, dest); saved.append(dest)
        print("  [doc]", dest)

print()
msg = f"Done. {len(saved)} document(s) saved."
if failed:
    msg += f" ({failed} failed)"
print(msg)

if saved:
    z = shutil.make_archive("/content/transcriptions", "zip", OUTPUT_DIR)
    from google.colab import files as _dl
    _dl.download(z)
